# Fase 5 · M01: Preparación de Datos para Modelado

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M01 — Preparación |

---

## 🎯 Qué hace

Prepara el dataset definitivo `dataset_final_tfm.parquet` (D_strict, 19 features + target `abandono`)
para el modelado clásico supervisado de la Fase 5. Establece particiones reproducibles y
construye el pipeline de preprocesamiento que usarán todos los módulos posteriores (M02–M06).

## 📋 Requisitos

- `data/automl/dataset_final_tfm.parquet` — producido por Fase 3 (auditoría leakage)
- `src/` con `config.py`, `utils/`, `html/`
- Entorno: `tfm_abandono` (scikit-learn, imbalanced-learn, joblib, pandas)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/X_train.parquet` | Features entrenamiento (80%) |
| `data/05_modelado/X_test.parquet` | Features test (20%) — **intocable hasta evaluación final** |
| `data/05_modelado/y_train.parquet` | Target entrenamiento |
| `data/05_modelado/y_test.parquet` | Target test |
| `data/05_modelado/pipeline_preprocesamiento.pkl` | Pipeline sklearn serializado |
| `data/05_modelado/meta_preparacion.json` | Metadatos: semilla, splits, columnas, distribución |
| `docs/html/fase5/m01_preparacion.html` | Informe HTML |

## 🔄 Flujo

```
dataset_final_tfm.parquet
    ↓ auditoría de entrada (nulos, tipos, distribución target)
    ↓ split estratificado 80/20 (random_state=42)
    ↓ pipeline: imputación → escalado → encoding
    ↓ verificación de proporciones target
    ↓ serialización splits + pipeline
    → X_train / X_test / y_train / y_test  +  pipeline_preprocesamiento.pkl
```

## ➡️ Siguiente

`f5_m02_lineales.ipynb` — Regresión Logística, LDA, SVM

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================
# Detecta ROOT ascendiendo hasta encontrar src/
# Importa utilidades del proyecto: config, utils, html
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')

# ----- ROOT detection -----
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(6):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))  # ROOT, no ROOT/src

# ----- Imports del proyecto -----
from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina

# ----- Rutas de la fase -----
RUTA_MODELADO   = ROOT / 'data' / '05_modelado'
RUTA_HTML_FASE5 = RUTA_HTML / 'fase5'
crear_directorios([RUTA_MODELADO, RUTA_HTML_FASE5])

# ----- Constantes -----
RANDOM_STATE = 42
TEST_SIZE    = 0.20
TARGET       = 'abandono'
fmt          = formato_numero_es

info_entorno()
print(f'\n📂 ROOT       : {ROOT}')
print(f'📂 MODELADO   : {RUTA_MODELADO}')
print(f'📂 HTML fase5 : {RUTA_HTML_FASE5}')
print(f'\n🎲 RANDOM_STATE = {RANDOM_STATE}')
print(f'✂️  TEST_SIZE    = {TEST_SIZE:.0%}')

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA Y AUDITORÍA DEL DATASET
# ============================================================================
# Lee dataset_final_tfm.parquet (D_strict) y verifica integridad antes del split
# ============================================================================

ruta_dataset = RUTA_AUTOML / 'dataset_final_tfm.parquet'
assert ruta_dataset.exists(), f'❌ Dataset no encontrado: {ruta_dataset}'

df = pd.read_parquet(ruta_dataset)

print('=' * 65)
print('AUDITORÍA DE ENTRADA — dataset_final_tfm.parquet (D_strict)')
print('=' * 65)

# Dimensiones
n_filas, n_cols = df.shape
print(f'\n📐 Dimensiones : {fmt(n_filas)} filas × {n_cols} columnas')
print(f'🎯 Target       : "{TARGET}"')

# Distribución del target
vc = df[TARGET].value_counts()
print(f'\n📊 Distribución target:')
print(f'   Continúa  (0): {fmt(vc.get(0, 0))}  ({vc.get(0, 0)/n_filas*100:.1f}%)')
print(f'   Abandona  (1): {fmt(vc.get(1, 0))}  ({vc.get(1, 0)/n_filas*100:.1f}%)')

# Nulos
nulos_total = df.isnull().sum().sum()
print(f'\n🔍 Nulos totales: {fmt(nulos_total)}')
if nulos_total > 0:
    cols_con_nulos = df.isnull().sum()
    cols_con_nulos = cols_con_nulos[cols_con_nulos > 0].sort_values(ascending=False)
    print('   Columnas con nulos:')
    for col, n in cols_con_nulos.items():
        print(f'   · {col}: {fmt(n)} ({n/n_filas*100:.2f}%)')
else:
    print('   ✅ Sin nulos')

# Tipos de datos
print(f'\n🗂️ Tipos de datos:')
tipos = df.dtypes.value_counts()
for t, c in tipos.items():
    print(f'   {str(t):<15} → {c} columna(s)')

# Features (todo menos el target)
features = [c for c in df.columns if c != TARGET]
print(f'\n📋 Features disponibles ({len(features)}):')
for f_ in features:
    dtype = df[f_].dtype
    nuniq = df[f_].nunique()
    print(f'   · {f_:<35}  dtype={str(dtype):<10}  nuniq={nuniq}')

AUDITORÍA DE ENTRADA — dataset_final_tfm.parquet (D_strict)

📐 Dimensiones : 33.621 filas × 20 columnas
🎯 Target       : "abandono"

📊 Distribución target:
   Continúa  (0): 23.788  (70.8%)
   Abandona  (1): 9.833  (29.2%)

🔍 Nulos totales: 12.740
   Columnas con nulos:
   · orden_preferencia: 4.699 (13.98%)
   · nota_selectividad: 3.118 (9.27%)
   · nota_acceso: 2.567 (7.64%)
   · nota_1er_anio: 2.353 (7.00%)
   · max_pagos: 3 (0.01%)

🗂️ Tipos de datos:
   int64           → 14 columna(s)
   float64         → 6 columna(s)

📋 Features disponibles (19):
   · cred_superados_anio_1er              dtype=float64     nuniq=316
   · nota_1er_anio                        dtype=float64     nuniq=473
   · nota_acceso                          dtype=float64     nuniq=866
   · rama                                 dtype=int64       nuniq=5
   · sexo                                 dtype=int64       nuniq=2
   · edad_entrada                         dtype=int64       nuniq=51
   · pais_nombre          

In [3]:
# ============================================================================
# CELDA 3: CLASIFICACIÓN DE FEATURES (numéricas vs. categóricas)
# ============================================================================
# Separa automáticamente numéricas y categóricas para el pipeline
# ============================================================================

X = df[features].copy()
y = df[TARGET].copy()

# Clasificar automáticamente
cols_numericas    = X.select_dtypes(include=['int64', 'int32', 'float64', 'float32']).columns.tolist()
cols_categoricas  = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print('=' * 65)
print('CLASIFICACIÓN DE FEATURES')
print('=' * 65)
print(f'\nNUMÉRICAS ({len(cols_numericas)}):')
for c in cols_numericas:
    print(f'   · {c}')

if cols_categoricas:
    print(f'\nCATEGÓRICAS ({len(cols_categoricas)}):')
    for c in cols_categoricas:
        print(f'   · {c}  →  {X[c].nunique()} valores únicos')
else:
    print(f'\nCATEGÓRICAS (0): todas las features son numéricas')

print(f'\n✅ Total features: {len(cols_numericas)} numéricas + {len(cols_categoricas)} categóricas = {len(features)} features')

CLASIFICACIÓN DE FEATURES

NUMÉRICAS (19):
   · cred_superados_anio_1er
   · nota_1er_anio
   · nota_acceso
   · rama
   · sexo
   · edad_entrada
   · pais_nombre
   · provincia
   · via_acceso
   · orden_preferencia
   · universidad_origen
   · tuvo_beca
   · n_anios_beca
   · situacion_laboral
   · nota_selectividad
   · max_pagos
   · indicador_interrupcion
   · anios_gap
   · anios_sin_beca

CATEGÓRICAS (0): todas las features son numéricas

✅ Total features: 19 numéricas + 0 categóricas = 19 features


In [4]:
# ============================================================================
# CELDA 4: SPLIT ESTRATIFICADO 80/20
# ============================================================================
# Split fijo con random_state=42. El test set NO SE TOCA hasta M06.
# stratify=y garantiza la misma proporción de abandonos en train y test
# ============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y
)

print('=' * 65)
print('SPLIT ESTRATIFICADO 80/20')
print('=' * 65)
print(f'\nTotal  : {fmt(len(X))} registros')
print(f'Train  : {fmt(len(X_train))} ({len(X_train)/len(X)*100:.1f}%)  →  abandonos: {fmt(y_train.sum())} ({y_train.mean()*100:.1f}%)')
print(f'Test   : {fmt(len(X_test))}  ({len(X_test)/len(X)*100:.1f}%)  →  abandonos: {fmt(y_test.sum())}  ({y_test.mean()*100:.1f}%)')
print()

# Verificar que las proporciones son similares (diferencia < 1 p.p.)
diff = abs(y_train.mean() - y_test.mean()) * 100
if diff < 1.0:
    print(f'✅ Stratify OK: diferencia en proporción target = {diff:.3f} p.p.')
else:
    print(f'⚠️  Diferencia en proporción target = {diff:.3f} p.p.  (> 1 p.p., revisar)')

print()
print('⚠️  ATENCIÓN: X_test / y_test son INTOCABLES hasta f5_m06_modelo_final.ipynb')

SPLIT ESTRATIFICADO 80/20

Total  : 33.621 registros
Train  : 26.896 (80.0%)  →  abandonos: 7.866 (29.2%)
Test   : 6.725  (20.0%)  →  abandonos: 1.967  (29.2%)

✅ Stratify OK: diferencia en proporción target = 0.003 p.p.

⚠️  ATENCIÓN: X_test / y_test son INTOCABLES hasta f5_m06_modelo_final.ipynb


In [5]:
# ============================================================================
# CELDA 5: CONSTRUCCIÓN DEL PIPELINE DE PREPROCESAMIENTO
# ============================================================================
# Pipeline sklearn robusto y reproducible:
#   · Numéricas : imputación mediana → StandardScaler
#   · Categóricas: imputación moda  → OrdinalEncoder
# Se ajusta SOLO sobre X_train (sin contaminación del test)
# ============================================================================

# Sub-pipeline numérico
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Sub-pipeline categórico
pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# ColumnTransformer — combina ambos sub-pipelines
transformador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_numericas),
        ('cat', pipe_cat, cols_categoricas)
    ] if cols_categoricas else [
        ('num', pipe_num, cols_numericas)
    ],
    remainder='passthrough'
)

# Ajustar SOLO sobre train
tracemalloc.start()
t0 = time.time()
X_train_prep = transformador.fit_transform(X_train)
X_test_prep  = transformador.transform(X_test)
t_fit = time.time() - t0
mem_actual, mem_pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Reconstruir como DataFrame con nombres de columnas
nombres_salida = cols_numericas + cols_categoricas
X_train_prep = pd.DataFrame(X_train_prep, columns=nombres_salida, index=X_train.index)
X_test_prep  = pd.DataFrame(X_test_prep,  columns=nombres_salida, index=X_test.index)

print('=' * 65)
print('PIPELINE DE PREPROCESAMIENTO')
print('=' * 65)
print(f'\nEstructura:')
print(f'  Numéricas  ({len(cols_numericas)}): imputación mediana → StandardScaler')
print(f'  Categóricas ({len(cols_categoricas)}): imputación moda   → OrdinalEncoder')
print(f'\nFit sobre X_train únicamente (sin data leakage)')
print(f'\nResultados:')
print(f'  X_train_prep: {X_train_prep.shape[0]:,} × {X_train_prep.shape[1]}')
print(f'  X_test_prep : {X_test_prep.shape[0]:,} × {X_test_prep.shape[1]}')
print(f'\n⏱️  Tiempo fit+transform : {t_fit:.3f} s')
print(f'💾 Memoria pico          : {mem_pico / 1024**2:.1f} MB')

PIPELINE DE PREPROCESAMIENTO

Estructura:
  Numéricas  (19): imputación mediana → StandardScaler
  Categóricas (0): imputación moda   → OrdinalEncoder

Fit sobre X_train únicamente (sin data leakage)

Resultados:
  X_train_prep: 26,896 × 19
  X_test_prep : 6,725 × 19

⏱️  Tiempo fit+transform : 0.205 s
💾 Memoria pico          : 21.2 MB


In [6]:
# ============================================================================
# CELDA 6: ANÁLISIS DE DESBALANCE DE CLASES
# ============================================================================
# Cuantifica el desbalance y documenta las 3 estrategias disponibles
# Los módulos M02–M05 elegirán la estrategia óptima para cada familia
# ============================================================================

print('=' * 65)
print('ANÁLISIS DE DESBALANCE DE CLASES')
print('=' * 65)

n_neg  = (y_train == 0).sum()
n_pos  = (y_train == 1).sum()
ratio  = n_neg / n_pos
pct_ab = y_train.mean() * 100

print(f'\nEn el conjunto de TRAIN:')
print(f'  Clase 0 (continúa)  : {fmt(n_neg)} ({100-pct_ab:.1f}%)')
print(f'  Clase 1 (abandona)  : {fmt(n_pos)} ({pct_ab:.1f}%)')
print(f'  Ratio mayoritaria/minoritaria: {ratio:.2f}:1')
print()

if ratio < 1.5:
    print('ℹ️  Dataset equilibrado: no se requiere ajuste especial')
elif ratio < 3:
    print('⚠️  Desbalance moderado: recomendable class_weight o SMOTE')
else:
    print('⚠️  Desbalance elevado: SMOTE o class_weight son necesarios')

print()
print('📋 Estrategias disponibles para M02–M05:')
print('   A) Sin ajuste          → baseline natural')
print('   B) class_weight=balanced → penaliza errores en clase minoritaria')
print('   C) SMOTE (en train)    → sobremuestreo sintético de la clase minoritaria')
print()
print('Cada módulo de modelado (M02–M05) seleccionará la estrategia')
print('con mejor F1/AUC mediante validación cruzada estratificada 5-fold.')

ANÁLISIS DE DESBALANCE DE CLASES

En el conjunto de TRAIN:
  Clase 0 (continúa)  : 19.030 (70.8%)
  Clase 1 (abandona)  : 7.866 (29.2%)
  Ratio mayoritaria/minoritaria: 2.42:1

⚠️  Desbalance moderado: recomendable class_weight o SMOTE

📋 Estrategias disponibles para M02–M05:
   A) Sin ajuste          → baseline natural
   B) class_weight=balanced → penaliza errores en clase minoritaria
   C) SMOTE (en train)    → sobremuestreo sintético de la clase minoritaria

Cada módulo de modelado (M02–M05) seleccionará la estrategia
con mejor F1/AUC mediante validación cruzada estratificada 5-fold.


In [7]:
# ============================================================================
# CELDA 7: VISUALIZACIONES
# ============================================================================
# 1) Distribución del target (barras) en total, train y test
# 2) Mapa de nulos por columna
# 3) Distribuciones numéricas antes/después del escalado (muestra 6 features)
# ============================================================================

COLOR_PRINCIPAL = '#3182ce'
COLOR_ABANDONO  = '#e53e3e'
COLOR_CONTINUA  = '#38a169'

graficos_b64 = {}

# ---- GRÁFICO 1: Distribución target ----
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
sets_info = [
    ('Total', y),
    ('Train (80%)', y_train),
    ('Test (20%)', y_test)
]
for ax, (titulo, serie) in zip(axes, sets_info):
    counts = serie.value_counts().sort_index()
    pcts   = serie.value_counts(normalize=True).sort_index() * 100
    bars   = ax.bar(
        ['Continúa (0)', 'Abandona (1)'],
        counts.values,
        color=[COLOR_CONTINUA, COLOR_ABANDONO],
        edgecolor='white',
        linewidth=1.5
    )
    for bar, pct in zip(bars, pcts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 30,
            f'{pct:.1f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_ylabel('Registros')
    ax.set_ylim(0, counts.max() * 1.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Distribución del target `abandono` en total, train y test', fontsize=13, y=1.02)
plt.tight_layout()
graficos_b64['distribucion_target'] = figura_a_base64(fig)
plt.close(fig)

# ---- GRÁFICO 2: Nulos por columna ----
nulos_por_col = X.isnull().sum().sort_values(ascending=False)
nulos_por_col = nulos_por_col[nulos_por_col >= 0]  # incluir todas (aunque sean 0)
pct_nulos     = (nulos_por_col / len(X) * 100).round(2)

fig, ax = plt.subplots(figsize=(10, max(4, len(features) * 0.4)))
colores = [COLOR_PRINCIPAL if v > 0 else '#cbd5e0' for v in nulos_por_col.values]
ax.barh(nulos_por_col.index, pct_nulos.values, color=colores, edgecolor='white')
ax.set_xlabel('% nulos')
ax.set_title('Porcentaje de valores nulos por feature', fontweight='bold')
ax.axvline(x=5, color='orange', linestyle='--', linewidth=1, label='Umbral 5%')
ax.axvline(x=20, color='red', linestyle='--', linewidth=1, label='Umbral 20%')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['nulos_por_columna'] = figura_a_base64(fig)
plt.close(fig)

# ---- GRÁFICO 3: Distribuciones antes/después escalado (6 numéricas) ----
n_mostrar = min(6, len(cols_numericas))
cols_muestra = cols_numericas[:n_mostrar]

fig, axes = plt.subplots(2, n_mostrar, figsize=(n_mostrar * 3.5, 7))
if n_mostrar == 1:
    axes = axes.reshape(2, 1)

for idx, col in enumerate(cols_muestra):
    # Antes del escalado
    axes[0, idx].hist(X_train[col].dropna(), bins=30, color=COLOR_PRINCIPAL, edgecolor='white', linewidth=0.5)
    axes[0, idx].set_title(col, fontsize=9)
    axes[0, idx].set_ylabel('Frec.' if idx == 0 else '')
    axes[0, idx].spines['top'].set_visible(False)
    axes[0, idx].spines['right'].set_visible(False)
    if idx == 0:
        axes[0, idx].set_ylabel('Antes escalado')

    # Después del escalado
    axes[1, idx].hist(X_train_prep[col].dropna(), bins=30, color='#805ad5', edgecolor='white', linewidth=0.5)
    axes[1, idx].spines['top'].set_visible(False)
    axes[1, idx].spines['right'].set_visible(False)
    if idx == 0:
        axes[1, idx].set_ylabel('Después escalado')

fig.suptitle('Distribuciones antes y después de StandardScaler (muestra 6 features)', fontsize=12, y=1.01)
plt.tight_layout()
graficos_b64['distribuciones_escalado'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Gráficos generados:', list(graficos_b64.keys()))

✅ Gráficos generados: ['distribucion_target', 'nulos_por_columna', 'distribuciones_escalado']


In [8]:
# ============================================================================
# CELDA 8: SERIALIZACIÓN — SPLITS + PIPELINE
# ============================================================================
# Guarda X_train, X_test, y_train, y_test como parquets
# Serializa el pipeline de preprocesamiento con joblib
# Exporta metadatos en JSON para trazabilidad
# ============================================================================

# Guardar splits (sin preprocesar — los notebooks M02–M05 aplican el pipeline)
X_train.to_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test.to_parquet(RUTA_MODELADO  / 'X_test.parquet')
y_train.to_frame().to_parquet(RUTA_MODELADO / 'y_train.parquet')
y_test.to_frame().to_parquet(RUTA_MODELADO  / 'y_test.parquet')

# Guardar también las versiones preprocesadas (útiles para modelos que no incluyen pipeline)
X_train_prep.to_parquet(RUTA_MODELADO / 'X_train_prep.parquet')
X_test_prep.to_parquet(RUTA_MODELADO  / 'X_test_prep.parquet')

# Serializar el pipeline
ruta_pipeline = RUTA_MODELADO / 'pipeline_preprocesamiento.pkl'
joblib.dump(transformador, ruta_pipeline)

# Metadatos de la preparación
meta = {
    'fecha_preparacion'   : datetime.now().isoformat(),
    'dataset_origen'      : 'dataset_final_tfm.parquet (D_strict)',
    'random_state'        : RANDOM_STATE,
    'test_size'           : TEST_SIZE,
    'n_total'             : int(len(X)),
    'n_train'             : int(len(X_train)),
    'n_test'              : int(len(X_test)),
    'pct_abandono_total'  : round(float(y.mean() * 100), 2),
    'pct_abandono_train'  : round(float(y_train.mean() * 100), 2),
    'pct_abandono_test'   : round(float(y_test.mean() * 100), 2),
    'n_features'          : len(features),
    'features'            : features,
    'cols_numericas'      : cols_numericas,
    'cols_categoricas'    : cols_categoricas,
    'nulos_en_dataset'    : int(nulos_total),
    'pipeline_pkl'        : str(ruta_pipeline.name),
    'archivos_generados'  : [
        'X_train.parquet', 'X_test.parquet',
        'y_train.parquet', 'y_test.parquet',
        'X_train_prep.parquet', 'X_test_prep.parquet',
        'pipeline_preprocesamiento.pkl',
        'meta_preparacion.json'
    ]
}

ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
with open(ruta_meta, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('=' * 65)
print('ARCHIVOS GUARDADOS')
print('=' * 65)
print(f'\n📁 Directorio: {RUTA_MODELADO}')
for archivo in meta['archivos_generados']:
    ruta_arch = RUTA_MODELADO / archivo
    tamano = ruta_arch.stat().st_size / 1024 if ruta_arch.exists() else 0
    print(f'   ✅ {archivo:<42} {tamano:>7.1f} KB')

ARCHIVOS GUARDADOS

📁 Directorio: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado
   ✅ X_train.parquet                              437.6 KB
   ✅ X_test.parquet                               122.2 KB
   ✅ y_train.parquet                              164.0 KB
   ✅ y_test.parquet                                41.0 KB
   ✅ X_train_prep.parquet                         441.6 KB
   ✅ X_test_prep.parquet                          128.5 KB
   ✅ pipeline_preprocesamiento.pkl                  3.4 KB
   ✅ meta_preparacion.json                          1.5 KB


In [9]:
# ============================================================================
# CELDA 9: GENERAR HTML
# ============================================================================
# Usa render_pagina (estándar del proyecto TFM desde Fase 5)
# El nombre del fichero es f5_m01_preparacion.ipynb → título inferido automáticamente
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_FASE5 / 'm01_preparacion.html'

# KPIs resumen
kpis_datos = [
    {'valor': fmt(len(X)),        'titulo': 'Registros totales'},
    {'valor': fmt(len(X_train)),  'titulo': 'Train (80%)'},
    {'valor': fmt(len(X_test)),   'titulo': 'Test (20%)'},
    {'valor': str(len(features)), 'titulo': 'Features'},
    {'valor': f'{y_train.mean()*100:.1f}%', 'titulo': 'Abandono train'},
    {'valor': f'{y_test.mean()*100:.1f}%',  'titulo': 'Abandono test'},
]

# Tabla de features
filas_features = ''.join(
    f'<tr><td>{col}</td><td>{str(X[col].dtype)}</td>'
    f'<td>{"Numérica" if col in cols_numericas else "Categórica"}</td>'
    f'<td>{X[col].isnull().sum()} ({X[col].isnull().mean()*100:.1f}%)</td></tr>'
    for col in features
)

tabla_features_html = f'''
<table class="tabla-datos">
  <thead>
    <tr><th>Feature</th><th>Tipo dato</th><th>Tipo feature</th><th>Nulos</th></tr>
  </thead>
  <tbody>
    {filas_features}
  </tbody>
</table>
'''

# Tabla de archivos generados
filas_archivos = ''.join(
    f'<tr><td><code>{a}</code></td></tr>'
    for a in meta['archivos_generados']
)
tabla_archivos_html = f'<table class="tabla-datos"><tbody>{filas_archivos}</tbody></table>'

# Contenido HTML
secciones_html = f'''
<section class="seccion">
  <h2>Dataset de entrada</h2>
  <p>Dataset <strong>D_strict</strong> ({fmt(len(X))} registros × {len(features)} features + target <code>abandono</code>).
  Producido por Fase 3 tras auditoría de data leakage. Sin contaminación del conjunto de test.</p>
  <div class="kpis-grid">
    {''.join(f'<div class="kpi"><span class="kpi-valor">{k["valor"]}</span><span class="kpi-titulo">{k["titulo"]}</span></div>' for k in kpis_datos)}
  </div>
</section>

<section class="seccion">
  <h2>Distribución del target</h2>
  <p>Validación de que el split estratificado mantiene proporciones de abandono consistentes en train y test.</p>
  <img src="data:image/png;base64,{graficos_b64['distribucion_target']}" style="max-width:100%">
</section>

<section class="seccion">
  <h2>Análisis de nulos</h2>
  <p>Porcentaje de valores nulos por feature. El pipeline de preprocesamiento aplica imputación por mediana (numéricas) o moda (categóricas).</p>
  <img src="data:image/png;base64,{graficos_b64['nulos_por_columna']}" style="max-width:100%">
</section>

<section class="seccion">
  <h2>Efecto del escalado</h2>
  <p>Comparación de distribuciones antes y después de aplicar <code>StandardScaler</code> sobre las variables numéricas.</p>
  <img src="data:image/png;base64,{graficos_b64['distribuciones_escalado']}" style="max-width:100%">
</section>

<section class="seccion">
  <h2>Features del modelo</h2>
  {tabla_features_html}
</section>

<section class="seccion">
  <h2>Archivos generados</h2>
  <p>Todos guardados en <code>data/05_modelado/</code></p>
  {tabla_archivos_html}
</section>

<section class="seccion">
  <h2>Estrategias de balance disponibles</h2>
  <p>El dataset presenta un ratio de desbalance <strong>{ratio:.2f}:1</strong> ({pct_ab:.1f}% abandono en train).
  Los módulos M02–M05 evaluarán las siguientes estrategias:</p>
  <ul>
    <li><strong>A) Sin ajuste</strong> — baseline natural, útil para comparación</li>
    <li><strong>B) class_weight=balanced</strong> — penaliza errores en clase minoritaria, sin alterar datos</li>
    <li><strong>C) SMOTE</strong> — sobremuestreo sintético aplicado solo sobre train (imbalanced-learn)</li>
  </ul>
  <p>La estrategia óptima se selecciona por F1 y AUC-ROC en validación cruzada 5-fold estratificada.</p>
</section>
'''

render_pagina(
    'f5_m01_preparacion.ipynb',
    secciones_html,
    RUTA_HTML_SALIDA,
    carpeta_notebook = 'fase5_modelado'
)

print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')


✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m01_preparacion.html


In [10]:
# ============================================================================
# CELDA 10: RESUMEN FINAL
# ============================================================================

print('=' * 65)
print('RESUMEN FINAL — f5_m01_preparacion')
print('=' * 65)
print()
print(f'Dataset    : dataset_final_tfm.parquet (D_strict)')
print(f'Registros  : {fmt(len(X))} ({fmt(len(X_train))} train / {fmt(len(X_test))} test)')
print(f'Features   : {len(features)} ({len(cols_numericas)} numéricas + {len(cols_categoricas)} categóricas)')
print(f'Target     : abandono  |  Abandono train={y_train.mean()*100:.1f}%  Test={y_test.mean()*100:.1f}%')
print(f'Desbalance : {ratio:.2f}:1  (clase 0 : clase 1)')
print(f'Semilla    : random_state={RANDOM_STATE}')
print()
print('📁 Archivos en data/05_modelado/:')
print('   X_train.parquet / X_test.parquet        → splits sin preprocesar')
print('   y_train.parquet / y_test.parquet')
print('   X_train_prep.parquet / X_test_prep.parquet → splits preprocesados')
print('   pipeline_preprocesamiento.pkl            → pipeline sklearn serializado')
print('   meta_preparacion.json                    → metadatos del split')
print()
print('📄 HTML: docs/html/fase5/m01_preparacion.html')
print()
print('➡️  Siguiente: f5_m02_lineales.ipynb')
print('   (Regresión Logística, LDA, SVM — cargan X_train.parquet / y_train.parquet)')

RESUMEN FINAL — f5_m01_preparacion

Dataset    : dataset_final_tfm.parquet (D_strict)
Registros  : 33.621 (26.896 train / 6.725 test)
Features   : 19 (19 numéricas + 0 categóricas)
Target     : abandono  |  Abandono train=29.2%  Test=29.2%
Desbalance : 2.42:1  (clase 0 : clase 1)
Semilla    : random_state=42

📁 Archivos en data/05_modelado/:
   X_train.parquet / X_test.parquet        → splits sin preprocesar
   y_train.parquet / y_test.parquet
   X_train_prep.parquet / X_test_prep.parquet → splits preprocesados
   pipeline_preprocesamiento.pkl            → pipeline sklearn serializado
   meta_preparacion.json                    → metadatos del split

📄 HTML: docs/html/fase5/m01_preparacion.html

➡️  Siguiente: f5_m02_lineales.ipynb
   (Regresión Logística, LDA, SVM — cargan X_train.parquet / y_train.parquet)
